# Pereira2018 Ridge Ceiling Computation

Pre-computes ceilings for the `Pereira2018.{243,384}sentences-ridge` benchmarks
and saves them as local `.nc` files in `ceilings/`.

The ridge benchmarks currently lack pre-computed ceilings, so the ceiling is
recomputed from scratch on every `score()` call. After running this notebook,
the benchmark loads from the local `.nc` files instead.

In [1]:
import logging
import pickle
import sys
import warnings
from pathlib import Path

logging.basicConfig(stream=sys.stdout, level=logging.DEBUG)
for shush_logger in ['botocore', 'boto3', 's3transfer', 'urllib3']:
    logging.getLogger(shush_logger).setLevel(logging.INFO)
warnings.filterwarnings('ignore')

In [2]:
from brainscore_language import load_dataset, load_metric
from brainscore_language.benchmarks.pereira2018.ceiling_packaging import ExtrapolationCeiling
from brainscore_core.supported_data_standards.brainio.packaging import write_netcdf

import brainscore_language.benchmarks.pereira2018 as _pereira_pkg
CEILING_DIR = Path(_pereira_pkg.__file__).parent / 'ceilings'
CEILING_DIR.mkdir(exist_ok=True)
print(f'Saving ceilings to: {CEILING_DIR}')

Saving ceilings to: /Users/kartik/Brain-Score 2026/language/brainscore_language/benchmarks/pereira2018/ceilings


In [3]:
def load_data(experiment: str, atlas: str = 'language'):
    """Load Pereira2018 data, matching _Pereira2018Experiment._load_data."""
    lang_data = load_dataset('Pereira2018.language')
    data = load_dataset(f'Pereira2018.{atlas}')
    data.coords['presentation'] = lang_data.coords['presentation']
    data = data.sel(experiment=experiment)
    data = data.dropna('neuroid')
    # Use a ridge-specific identifier so the @store cache in
    # ExtrapolationCeiling.collect does not collide with linear ceilings.
    data.attrs['identifier'] = f'Pereira2018.language.{experiment}.ridge'
    return data

In [4]:
def compute_ceiling(experiment: str):
    """Compute the ridge ceiling, with pickle caching to survive notebook restarts."""
    cache_path = CEILING_DIR / f'ceiling_{experiment}_ridge.pkl'
    if cache_path.exists():
        print(f'Loading cached ceiling from {cache_path}')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    data = load_data(experiment)
    metric = load_metric('ridge_pearsonr', crossvalidation_kwargs=dict(
        split_coord='story',
        kfold='group',
        random_state=1234,
    ))

    ceiler = ExtrapolationCeiling()
    ceiling = ceiler(assembly=data, metric=metric)
    print(f'{experiment} ceiling: {float(ceiling):.4f}')

    # Cache the ceiling object so we don't redo the 68-min extrapolation
    with open(cache_path, 'wb') as f:
        pickle.dump(ceiling, f)
    print(f'Cached ceiling to {cache_path}')

    return ceiling

In [5]:
def save_ceiling_nc(ceiling, experiment: str):
    """Save all three nesting levels as .nc files for the benchmark to load."""
    identifier = f'Pereira2018.{experiment}-ridge'

    for prefix, assembly in [
        ('ceiling_', ceiling),
        ('ceiling_raw_', ceiling.raw),
        ('ceiling_raw_raw_', ceiling.raw.raw),
    ]:
        filename = f"{prefix}{identifier.replace('.', '_')}.nc"
        path = CEILING_DIR / filename
        write_netcdf(assembly, path)
        print(f'  Saved {path.name} ({path.stat().st_size / 1e6:.1f} MB)')

## 384sentences

In [6]:
ceiling_384 = compute_ceiling('384sentences')
save_ceiling_nc(ceiling_384, '384sentences')

DEBUG:brainscore_core.plugin_management.import_plugin:Plugin pereira2018 has no requirements file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/requirements.txt or setup file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/setup.py
DEBUG:brainscore_core.plugin_management.import_plugin:Plugin pereira2018 has no requirements file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/requirements.txt or setup file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/setup.py
DEBUG:result_caching._DiskStorage:Loading from storage: brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling.collect/identifier=Pereira2018.language.384sentences.ridge


neuroid extrapolations: 100%|██████████| 12155/12155 [1:08:22<00:00,  2.96it/s]

DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging neuroid ceilings


DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging bootstrapped_params
DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging error_low
DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging error_high
DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging endpoint_x
384sentences ceiling: 0.1352
Cached ceiling to /Users/kartik/Brain-Score 2026/language/brainscore_language/benchmarks/pereira2018/ceilings/ceiling_384sentences_ridge.pkl
DEBUG:brainscore_core.supported_data_standards.brainio.packaging:Writing assembly to /Users/kartik/Brain-Score 2026/language/brainscore_language/benchmarks/pereira2018/ceilings/ceiling_Pereira2018_384sentences-ridge.nc
  Saved ceiling_Pereira2018_384sentences-ridge.nc (0.0 MB)
DEBUG:brainscore_core.supported_data_standards.brainio.packaging:Writing assembly to /Users/kartik/Brain-Score 2026/

## 243sentences

In [7]:
ceiling_243 = compute_ceiling('243sentences')
save_ceiling_nc(ceiling_243, '243sentences')

DEBUG:brainscore_core.plugin_management.import_plugin:Plugin pereira2018 has no requirements file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/requirements.txt or setup file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/setup.py
DEBUG:brainscore_core.plugin_management.import_plugin:Plugin pereira2018 has no requirements file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/requirements.txt or setup file /Users/kartik/Brain-Score 2026/language/brainscore_language/data/pereira2018/setup.py
DEBUG:result_caching._DiskStorage:Running function: brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling.collect/identifier=Pereira2018.language.243sentences.ridge
DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Collecting data for extrapolation


num subjects:   0%|          | 0/5 [00:00<?, ?it/s]


INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:X has shape torch.Size([213, 1316]), y has shape torch.Size([213, 1325])
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Data preprocessing complete
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:gcv mode is eigen
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Iterating alphas
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Best alpha selected: 1000.0


INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:X has shape torch.Size([211, 1316]), y has shape torch.Size([211, 1325])
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Data preprocessing complete
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:gcv mode is eigen
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Iterating alphas
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Best alpha selected: 1000.0


INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:X has shape torch.Size([213, 1316]), y has shape torch.Size([213, 1325])
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Data preprocessing complete
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:gcv mode is eigen
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Iterating alphas
INFO:brainscore_language.metrics.linear_predictivity.ridgecv_gpu:Best alpha selected: 1000.0


...


num subjects: 100%|██████████| 5/5 [21:24<00:00, 256.82s/it]


DEBUG:result_caching._DiskStorage:Saving to storage: brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling.collect/identifier=Pereira2018.language.243sentences.ridge


neuroid extrapolations: 100%|██████████| 8031/8031 [38:49<00:00,  3.45it/s]

DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging neuroid ceilings


DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging bootstrapped_params
DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging error_low
DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging error_high
DEBUG:brainscore_language.benchmarks.pereira2018.ceiling_packaging.ExtrapolationCeiling:Merging endpoint_x
243sentences ceiling: 0.1337
Cached ceiling to /Users/kartik/Brain-Score 2026/language/brainscore_language/benchmarks/pereira2018/ceilings/ceiling_243sentences_ridge.pkl
DEBUG:brainscore_core.supported_data_standards.brainio.packaging:Writing assembly to /Users/kartik/Brain-Score 2026/language/brainscore_language/benchmarks/pereira2018/ceilings/ceiling_Pereira2018_243sentences-ridge.nc
  Saved ceiling_Pereira2018_243sentences-ridge.nc (0.0 MB)
DEBUG:brainscore_core.supported_data_standards.brainio.packaging:Writing assembly to /Users/kartik/Brain-Score 2026/

In [ ]:
from brainscore_language import score

model = 'oasm-sigma0.6'
benchmark = 'Pereira2018.384sentences-ridge'

In [11]:
score = score(model, benchmark)

/opt/anaconda3/envs/language-2026/lib/python3.11/site-packages/xarray/core/concat.py:500: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))
cross-validation: 100%|██████████| 10/10 [00:04<00:00,  2.01it/s]


In [12]:
score

<xarray.Score ()>
array(0.)
Attributes:
    layer_scores:          {'oasm_sigma0.6': <xarray.Score ()>\narray(0.)\nAt...
    raw:                   <xarray.Score ()>\narray(0.)
    ceiling:               <xarray.Score 'data' ()>\narray(0.1352125)\nAttrib...
    model_identifier:      oasm-sigma0.6
    benchmark_identifier:  Pereira2018.384sentences-ridge